# Missing values

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/Andrey/blob/main/examples/missing_values.ipynb)

A method that reads only the rows where a test's variables are observed (test-wise deletion) is
valid when values are missing completely at random: whether a value is missing depends on no value
in the data. When it depends on another variable, those rows are a selected sample and the method
can keep edges the data do not support. Here we remove values from simulated data in a known way,
fit PC with its default handling and with `missing="mvpc"`, fit GES with `missing="em"`, and compare
each graph with the true one. Four more cases follow: values missing not at random, FCI,
categorical data, and a time series.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.19.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.19.0" pandas

In [2]:
import numpy as np
import pandas as pd

import andrey
import andrey.viz
from andrey import data
from andrey.data import missingness
from andrey.metrics import shd, temporal_scores, to_cpdag

## Data with a known graph

`data.sample_scm` draws 2,000 rows of ten variables from a linear-Gaussian model whose graph we
know. PC and GES can recover that graph only up to its Markov equivalence class, so we score each
result against the true CPDAG.

In [3]:
complete = data.sample_scm(d=10, n=2000, seed=9, density=2)
truth = to_cpdag(complete.graph)
andrey.viz.draw(complete.graph)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 356" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="356" fill="#fafbfc"/><line x1="109.35" y1="116.72" x2="44.80" y2="105.27" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.20,118.47 110.23,111.80 108.48,121.65" fill="#3a4658" stroke-linejoin="round"/><line x1="114.38" y1="135.52" x2="41.96" y2="196.26" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="122.04,129.10 111.16,131.69 117.59,139.35" fill="#3a4658" stroke-linejoin="round"/><path d="M 44.00 108.01 Q 224.00 183.11 409.00 186.45" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.00,186.64 409.09,181.46 408.91,191.45" fill="#3a4658" stroke-linejoin="round"/><line x1="40.35" y1="214.58" x2="117.23" y2="306.34" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.65,314.00 121.06,303.13 113.40,309.55" fill="#3a4658" stroke-linejoin="round"/><line x1="44.99" y1="204.04" x2="409.02" y2="187.89" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.01,187.45 408.80,182.90 409.24,192.89" fill="#3a4658" stroke-linejoin="round"/><line x1="43.70" y1="210.28" x2="111.30" y2="242.98" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="120.30,247.34 113.47,238.48 109.12,247.48" fill="#3a4658" stroke-linejoin="round"/><line x1="144.80" y1="321.70" x2="209.35" y2="310.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.20,308.50 208.48,305.32 210.23,315.17" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="323.97" x2="45.00" y2="323.97" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,323.97 109.00,318.97 109.00,328.97" fill="#3a4658" stroke-linejoin="round"/><line x1="414.38" y1="201.65" x2="341.96" y2="262.39" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="422.04,195.22 411.16,197.82 417.59,205.48" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="278.43" x2="244.25" y2="301.88" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,275.09 308.65,273.72 312.00,283.15" fill="#3a4658" stroke-linejoin="round"/><circle cx="132.00" cy="120.74" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="120.74" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="32.00" cy="103.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="32.00" cy="32.03" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="32.03" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="204.61" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="204.61" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="132.00" cy="323.97" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="323.97" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="432.00" cy="186.87" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="186.87" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">5</text><circle cx="132.00" cy="253.00" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="253.00" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">6</text><circle cx="332.00" cy="270.74" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332

## Remove values

`missingness.mar(0.3, form="quantile")` is missing at random given the observed variables: a
variable that can go missing has one parent among the variables that never do, and its value is
missing with probability 0.9 when the parent is below a quantile of its values and 0.01 otherwise.
`missingness.draw` applies the rule to the complete data and returns the mask and each indicator's
parents.

In [4]:
rule = missingness.mar(0.3, form="quantile")
mask = missingness.draw(complete.data, rule, seed=9)
gappy = np.where(mask.mask, np.nan, complete.data)
print("share missing, by column:", np.round(mask.mask.mean(axis=0), 2).tolist())
print("parent of each missingness indicator:", mask.parents)

share missing, by column: [0.29, 0.0, 0.0, 0.3, 0.0, 0.3, 0.3, 0.3, 0.3, 0.3]
parent of each missingness indicator: {0: [2], 3: [4], 5: [1], 6: [2], 7: [4], 8: [1], 9: [2]}


## PC with its default handling

The default, `missing=None`, is test-wise deletion for PC: each test reads the rows where its
variables are observed, and `metadata["rows_tested"]` gives the fewest rows and the median number of
rows a test read. The rows a test keeps are a selected sample, as they depend on the variables
through the parents above.

In [5]:
default = andrey.pc(gappy)
print("rows read by a test:", default.metadata["rows_tested"])
print("errors against the true CPDAG:", shd(default, truth))

andrey.pc: the data hold missing values and missing=None, so the fit ran the default handling, test-wise deletion: each test read only the rows where its variables are observed. Test-wise deletion is valid when values are missing completely at random, which means that whether a value is missing depends on no value in the data; the rows a test reads are then a random sample of all rows, and otherwise they are a selected sample that more rows do not correct. Pass missing="testwise" to choose it by name, and read the rows the tests kept in metadata["rows_tested"]. This notice is logged once per process.


rows read by a test: {'min': 645, 'median': 950.0}
errors against the true CPDAG: 5


## PC with `missing="mvpc"`

Missing-Value PC finds the variables each missingness indicator depends on, then tests again the
edges such an indicator could have kept, on data rebuilt from the rows where those variables are
observed.

In [6]:
mvpc = andrey.pc(gappy, missing="mvpc")
print("parents of the indicators it found:", mvpc.metadata["missingness_parents"])
print("edges it tested again and removed:", mvpc.metadata["corrected_edges"]["removed"])
print("errors against the true CPDAG:", shd(mvpc, truth))

parents of the indicators it found: {0: [2], 3: [4], 5: [0, 1], 6: [2], 7: [4], 8: [1], 9: [2]}
edges it tested again and removed: [[3, 9]]
errors against the true CPDAG: 1


## A score search with `missing="em"`

A search that reads the data only through their covariance cannot delete rows per test, so it takes
`missing="em"`: structural EM replaces each row's missing values by their expected values under the
model of the current graph and adds their uncertainty, which gives the expected covariance of the
data, and the search runs on that covariance over every row.

In [7]:
em = andrey.ges(gappy, missing="em")
print("errors against the true CPDAG:", shd(em, truth))

errors against the true CPDAG: 0


## Compare

PC and GES on the complete data show what each method makes of these data when nothing is missing.

In [8]:
outputs = {
    "PC, complete data": andrey.pc(complete.data),
    "GES, complete data": andrey.ges(complete.data),
    "PC, default": default,
    'PC, missing="mvpc"': mvpc,
    'GES, missing="em"': em,
}
pd.Series({name: shd(out, truth) for name, out in outputs.items()}, name="errors (SHD)").to_frame()

,errors (SHD)
"PC, complete data",1
"GES, complete data",0
"PC, default",5
"PC, missing=""mvpc""",1
"GES, missing=""em""",0


## Ten datasets

One draw can differ from the next. The same comparison on ten datasets, each with its own graph and
its own missing values:

In [9]:
rows = []
for seed in range(10):
    full = data.sample_scm(d=10, n=2000, seed=seed, density=2)
    gaps = missingness.draw(full.data, rule, seed=seed)
    x = np.where(gaps.mask, np.nan, full.data)
    true = to_cpdag(full.graph)
    rows.append(
        {
            "PC, complete data": shd(andrey.pc(full.data), true),
            "GES, complete data": shd(andrey.ges(full.data), true),
            "PC, default": shd(andrey.pc(x), true),
            'PC, missing="mvpc"': shd(andrey.pc(x, missing="mvpc"), true),
            'GES, missing="em"': shd(andrey.ges(x, missing="em"), true),
        }
    )
pd.DataFrame(rows).mean().round(1).to_frame("mean errors (SHD)")

,mean errors (SHD)
"PC, complete data",1.8
"GES, complete data",1.5
"PC, default",4.8
"PC, missing=""mvpc""",3.6
"GES, missing=""em""",1.1


On this dataset, PC's default handling makes 5 errors where PC on the complete data makes 1;
`missing="mvpc"` removes the edge it traces to the missingness parents and is back to 1, and GES
with `missing="em"` makes none, as it does on the complete data. Over ten datasets PC makes 4.8
errors by default and 3.6 with MVPC against 1.8 on the complete data, so MVPC closes part of the
gap, while GES with EM makes 1.1 against 1.5 on the complete data. The two searches differ in more
than how they treat gaps, so read each against its own fit on the complete data.

## Values missing not at random

`missingness.mnar` makes each indicator depend on a variable that has missing values itself, so
the reason a value is missing is not always observed. MVPC allows for this under its conditions,
among them that no value causes its own missingness. EM assumes that the reason is observed, so it
has no guarantee here.

In [10]:
not_at_random = missingness.mnar(0.3, form="quantile")
hidden = missingness.draw(complete.data, not_at_random, seed=9)
print("parent of each missingness indicator:", hidden.parents)
rows = []
for seed in range(10):
    full = data.sample_scm(d=10, n=2000, seed=seed, density=2)
    gaps = missingness.draw(full.data, not_at_random, seed=seed)
    x = np.where(gaps.mask, np.nan, full.data)
    true = to_cpdag(full.graph)
    rows.append(
        {
            "PC, complete data": shd(andrey.pc(full.data), true),
            "GES, complete data": shd(andrey.ges(full.data), true),
            "PC, default": shd(andrey.pc(x), true),
            'PC, missing="mvpc"': shd(andrey.pc(x, missing="mvpc"), true),
            'GES, missing="em"': shd(andrey.ges(x, missing="em"), true),
        }
    )
pd.DataFrame(rows).mean().round(1).to_frame("mean errors (SHD)")

parent of each missingness indicator: {0: [6], 3: [9], 5: [3], 6: [5], 7: [8], 8: [3], 9: [5]}


,mean errors (SHD)
"PC, complete data",1.8
"GES, complete data",1.5
"PC, default",3.6
"PC, missing=""mvpc""",3.0
"GES, missing=""em""",2.4


Over these ten datasets PC makes 3.6 errors by default and 3.0 with MVPC, against 1.8 on the
complete data. GES with EM makes 2.4 against 1.5 on the complete data. With values missing at
random it made 1.1, so here EM no longer matches the fit on the complete data.

## FCI

FCI deletes rows per test too, and `missing="testwise"` is its only handling. It is valid when
values are missing completely at random, as in every method: each test selects its own rows, so
FCI's PAG, which allows for one selection of the rows, does not account for them. Here we fit
the data with values missing at random from above and count the differences from the PAG of
the true graph, which has no selection, then repeat that on the ten datasets.

In [11]:
pag = complete.graph.to_pag()
fci = andrey.fci(gappy)
print("handling:", fci.metadata["missing"])
print("rows read by a test:", fci.metadata["rows_tested"])
print("differences from the true PAG, complete data:", shd(andrey.fci(complete.data), pag))
print("differences from the true PAG, missing values:", shd(fci, pag))
rows = []
for seed in range(10):
    full = data.sample_scm(d=10, n=2000, seed=seed, density=2)
    gaps = missingness.draw(full.data, rule, seed=seed)
    x = np.where(gaps.mask, np.nan, full.data)
    true = full.graph.to_pag()
    rows.append(
        {
            "FCI, complete data": shd(andrey.fci(full.data), true),
            "FCI, default": shd(andrey.fci(x), true),
        }
    )
pd.DataFrame(rows).mean().round(1).to_frame("mean differences (SHD)")

handling: testwise
rows read by a test: {'min': 645, 'median': 950.0}
differences from the true PAG, complete data: 1
differences from the true PAG, missing values: 6


,mean differences (SHD)
"FCI, complete data",2.9
"FCI, default",5.7


On this dataset FCI's PAG differs from the true one in 6 places with missing values and in 1 on the
complete data, and over ten datasets in 5.7 against 2.9. FCI has no correction for these edges: it
keeps them, as PC's default does.

## Categorical data

Test-wise deletion works with every test, the categorical ones included, while `"mvpc"` and `"em"`
need continuous data. `data.sample_discrete` draws ten categorical variables, and
`missingness.mcar(0.2)` removes a fifth of the values completely at random, the case test-wise
deletion is valid for. Dropping every incomplete row is valid there too, but it keeps few rows.

In [12]:
columns = list(range(10))
rows = []
for seed in range(10):
    full = data.sample_discrete(d=10, n=2000, seed=seed, density=2)
    gaps = missingness.draw(full.data, missingness.mcar(0.2), seed=seed)
    x = np.where(gaps.mask, np.nan, full.data)
    kept = x[~np.isnan(x).any(axis=1)]
    true = to_cpdag(full.graph)
    rows.append(
        {
            "PC, complete data": shd(andrey.pc(full.data, categorical=columns), true),
            "PC, default": shd(andrey.pc(x, categorical=columns), true),
            "PC, incomplete rows dropped": shd(andrey.pc(kept, categorical=columns), true),
            "complete rows": len(kept),
        }
    )
pd.DataFrame(rows).mean().round(1).to_frame("mean over ten datasets")

,mean over ten datasets
"PC, complete data",4.7
"PC, default",4.6
"PC, incomplete rows dropped",7.9
complete rows,210.6


With a fifth of the values missing, about 211 of the 2,000 rows are complete. PC with test-wise
deletion makes 4.6 errors, as many as on the complete data (4.7), and 7.9 on the complete rows
alone.

## A time series

A time series cannot drop a step without joining the steps around it. PCMCI keeps every step
instead: each of its tests reads the rows of the lagged design where all of its lagged variables
are observed. Here a tenth of the values of a four-variable series go missing completely at random,
and we count the wrong links from the step before.

In [13]:
rows = []
for seed in range(10):
    series = data.sample_scm(
        functional="var", noise="uniform", d=4, n=1000, seed=seed, density=1.0, scale="raw"
    )
    gaps = missingness.draw(series.data, missingness.mcar(0.1), seed=seed)
    x = np.where(gaps.mask, np.nan, series.data)
    out = andrey.pcmci(x, tau_max=1)
    rows.append(
        {
            "PCMCI, complete data": temporal_scores(
                andrey.pcmci(series.data, tau_max=1), series.graph
            )["lagged"]["shd"],
            "PCMCI, default": temporal_scores(out, series.graph)["lagged"]["shd"],
            "fewest rows read by a test": out.metadata["rows_tested"]["min"],
        }
    )
pd.DataFrame(rows).mean().round(1).to_frame("mean over ten series")

,mean over ten series
"PCMCI, complete data",2.1
"PCMCI, default",1.9
fewest rows read by a test,449.6


PCMCI gets 1.9 links wrong on average, against 2.1 on the complete series, although the test with
the fewest rows read about 450 of the 1,000 steps.

## Which handling to use

The [Missing values](https://andrey.abel.ai/docs/guides/missing-values.html) guide has a table from
your data and the reason values are missing to the `missing` value to use, and the
[0.16.0](https://andrey.abel.ai/docs/benchmarks/0.16.0.html#what-we-measured-on-missing-data) and
[0.17.0](https://andrey.abel.ai/docs/benchmarks/0.17.0.html#missing-data) release cards give what
we measured against reference implementations.